In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
response_OUS.isna().sum().sum()

0

In [4]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D3
clinical_train.isnull().sum().sum()

0

In [7]:
clinical_train

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,event_DFS
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370,32.909589,0
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699,27.715068,0
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414,43.101370,1
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899,17.589041,1
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395,47.473973,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361,41.490411,0
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630,36.821918,0
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522,40.043836,0
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705,40.964384,0


## Test dataset: MAASTRO 

In [8]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [9]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [10]:
rows_with_nan = response_MAASTRO[response_MAASTRO.isna().any(axis=1)]

print("Rows with NaN values:")
print(rows_with_nan)

Rows with NaN values:
    patient_id  OS  OS_event  LRC  LRC_event  DFS  DFS_event
10          11 NaN       NaN  NaN        NaN  NaN        NaN
20          21 NaN       NaN  NaN        NaN  NaN        NaN
31          32 NaN       NaN  NaN        NaN  NaN        NaN
51          52 NaN       NaN  NaN        NaN  NaN        NaN
83          84 NaN       NaN  NaN        NaN  NaN        NaN
86          87 NaN       NaN  NaN        NaN  NaN        NaN


In [11]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)

Numbers from 1 to 114 not present in data: []


In [12]:
MAASTRO_D3

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,1,55,0,0,1,0,0,1,1,1,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,2,55,0,0,1,0,0,0,0,0,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,3,55,0,0,1,0,0,0,0,1,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,4,61,1,0,0,0,1,1,0,1,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,6,70,0,0,1,0,0,1,1,1,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,111,63,0,0,0,0,1,0,0,1,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,112,63,0,0,1,0,0,1,1,1,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,113,54,0,0,1,0,0,1,1,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [13]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(MAASTRO_D3['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: [5, 9, 11, 21, 32, 36, 46, 52, 65, 76, 78, 81, 84, 87, 92]


In [14]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: []


In [15]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [16]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [17]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [18]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [19]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [20]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 388)
y_train:  (139,)


(99, 390)

# Feature Selection: RENT

In [21]:
selected_features = [
"shape_Sphericity",
"hpv_related",
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16",
"LBP_102_PET",
"shape_Flatness",
"shape_Elongation",
"uicc8_III-IV",
"shape_MajorAxisLength",
"glrlm_HighGrayLevelRunEmphasis_PET_c04"]

# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [22]:
original_X = X.copy()

In [23]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [24]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [25]:
X_new

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,0.761164,0.0,0.029425,0.000000,0.535140,0.600926,0.0,42.073251,16.969770
1,0.697049,0.0,0.037915,0.000000,0.367109,0.841579,0.0,24.613845,15.598394
2,0.565792,0.0,0.008009,0.000034,0.597785,0.772821,1.0,48.030294,17.334294
3,0.684364,0.0,0.018398,0.000000,0.405730,0.847727,0.0,25.589900,14.009277
4,0.503142,0.0,0.013051,0.000199,0.442406,0.831483,0.0,34.684750,21.202180
...,...,...,...,...,...,...,...,...,...
134,0.742102,1.0,0.014938,0.000000,0.523608,0.680294,0.0,33.069705,16.110383
135,0.722918,1.0,0.011441,0.000000,0.735524,0.758193,1.0,41.043692,21.249575
136,0.652963,1.0,0.020431,0.000000,0.648063,0.770113,0.0,36.618802,14.873884
137,0.724255,1.0,0.019663,0.000000,0.492193,0.628897,1.0,45.870392,21.648860


In [26]:
X_new_std

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,1.075042,0.0,1.274405,-0.562251,0.072232,-0.681983,0.0,-0.098422,0.081641
1,0.242767,0.0,2.273964,-0.562251,-1.302158,1.006304,0.0,-1.209119,-0.395928
2,-1.461057,0.0,-1.246890,-0.202838,0.584632,0.523938,1.0,0.280542,0.208583
3,0.078112,0.0,-0.023759,-0.562251,-0.986261,1.049432,0.0,-1.147026,-0.949323
4,-2.274305,0.0,-0.653294,1.514080,-0.686277,0.935477,0.0,-0.568449,1.555538
...,...,...,...,...,...,...,...,...,...
134,0.827589,1.0,-0.431120,-0.562251,-0.022089,-0.125182,0.0,-0.671191,-0.217633
135,0.578577,1.0,-0.842806,-0.562251,1.711257,0.421312,1.0,-0.163918,1.572043
136,-0.329498,1.0,0.215575,-0.562251,0.995873,0.504939,0.0,-0.445412,-0.648232
137,0.595927,1.0,0.125178,-0.562251,-0.279046,-0.485754,1.0,0.143137,1.711090


In [27]:
MAASTRO_new

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,0.668072,1,0.010495,0.000026,0.610062,0.765178,0,50.002093,19.034156
1,0.669961,0,0.035018,0.000167,0.504616,0.776540,1,41.753334,11.392444
2,0.624081,0,0.012819,0.000057,0.478604,0.697164,1,44.375483,14.567421
3,0.577624,0,0.029974,0.000000,0.446059,0.574636,1,46.115989,13.477331
4,0.630933,1,0.013458,0.000000,0.480378,0.633419,0,54.394967,16.365554
...,...,...,...,...,...,...,...,...,...
94,0.671754,0,0.039092,0.000000,0.577884,0.882411,1,34.218615,17.492295
95,0.632189,0,0.015831,0.000000,0.455642,0.535802,1,51.046869,14.267900
96,0.645548,1,0.011031,0.000000,0.631485,0.716610,1,50.417953,12.143752
97,0.727488,1,0.019801,0.000000,0.628338,0.665145,0,44.901412,15.300229


In [28]:
MAASTRO_new_std

,shape_Sphericity,hpv_related,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,LBP_102_PET,shape_Flatness,shape_Elongation,uicc8_III-IV,shape_MajorAxisLength,glrlm_HighGrayLevelRunEmphasis_PET_c04
0,-0.133381,1,-0.954258,-0.288893,0.685053,0.470316,0,0.405980,0.800544
1,-0.108854,0,1.932841,1.182201,-0.177437,0.550025,1,-0.118774,-1.860610
2,-0.704414,0,-0.680666,0.033974,-0.390198,-0.006831,1,0.048037,-0.754955
3,-1.307469,0,1.338998,-0.562251,-0.656393,-0.866418,1,0.158761,-1.134568
4,-0.615466,1,-0.605333,-0.562251,-0.375685,-0.454030,0,0.685437,-0.128772
...,...,...,...,...,...,...,...,...,...
94,-0.085581,0,2.412550,-0.562251,0.421853,1.292755,1,-0.598102,0.263605
95,-0.599161,0,-0.326069,-0.562251,-0.578011,-1.138850,1,0.472444,-0.859260
96,-0.425762,1,-0.891107,-0.562251,0.860275,0.129595,1,0.432435,-1.598975
97,0.637891,1,0.141405,-0.562251,0.834539,-0.231458,0,0.081495,-0.499761


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [29]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-22 02:38:37,858] A new study created in memory with name: no-name-680b2ea9-9c52-4326-a1ce-b8ea4312af19


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-22 02:38:41,727] A new study created in memory with name: no-name-3e24e95d-5d3b-4bb8-809a-f8721811c38b


Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.759656652360515
[I 2024-04-22 02:38:41,717] Trial 0 finished with value: 0.72444161978659 and parameters: {}. Best is trial 0 with value: 0.72444161978659.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.72444161978659], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 37, 919489), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 41, 717502), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.72444161978659


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.22218111763385023
Fold 2 IBS: 0.18214519470787685
Fold 3 IBS: 0.19943058215780188
Fold 4 IBS: 0.2249881896341225
Fold 5 IBS: 0.17079109178266783
[I 2024-04-22 02:38:42,006] Trial 0 finished with value: 0.19990723518326386 and parameters: {}. Best is trial 0 with value: 0.19990723518326386.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19990723518326386], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 41, 764006), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 42, 6773), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19990723518326386


In [30]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.724
train_ibs:  0.2


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.562
IBS score: 0.29


In [34]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [35]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-22 02:38:42,184] A new study created in memory with name: no-name-2069cc69-37ff-4e7f-9150-06ec6ee3aaff


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.687984496124031


[I 2024-04-22 02:38:42,366] A new study created in memory with name: no-name-b353f750-ff02-41c0-b21b-481f53d9bc02


Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6759656652360515
[I 2024-04-22 02:38:42,361] Trial 0 finished with value: 0.6487951905080959 and parameters: {}. Best is trial 0 with value: 0.6487951905080959.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6487951905080959], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 42, 223786), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 42, 361797), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6487951905080959


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709395077762
Fold 2 IBS: 0.2320398710570518
Fold 3 IBS: 0.22898186727039924
Fold 4 IBS: 0.24197476010091573
Fold 5 IBS: 0.2293955839675724
[I 2024-04-22 02:38:42,545] Trial 0 finished with value: 0.23592783526934333 and parameters: {}. Best is trial 0 with value: 0.23592783526934333.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783526934333], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 42, 401852), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 42, 544812), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783526934333


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.649
train_ibs:  0.236


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.537


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-22 02:38:42,698] A new study created in memory with name: no-name-66faff4f-994a-4d46-8563-6168b12e8155


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-22 02:38:43,010] Trial 0 finished with value: 0.7253468077103257 and parameters: {}. Best is trial 0 with value: 0.7253468077103257.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7253468077103257], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 42, 749705), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 43, 10071), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7253468077103257


[I 2024-04-22 02:38:43,017] A new study created in memory with name: no-name-3f287d3a-0196-400d-af7f-115e5a37602b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.22246158311007141
Fold 2 IBS: 0.18207907101733628
Fold 3 IBS: 0.19834248653978906
Fold 4 IBS: 0.22386774528137604
Fold 5 IBS: 0.16849199829394998
[I 2024-04-22 02:38:43,343] Trial 0 finished with value: 0.19904857684850458 and parameters: {}. Best is trial 0 with value: 0.19904857684850458.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19904857684850458], datetime_start=datetime.datetime(2024, 4, 22, 2, 38, 43, 46763), datetime_complete=datetime.datetime(2024, 4, 22, 2, 38, 43, 343755), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19904857684850458


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.725
train_ibs:  0.199


#### Test 

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.561


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.289


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-22 02:38:43,543] A new study created in memory with name: no-name-29bcb60f-f875-4e96-8340-dd802b09beeb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.6978723404255319
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-22 02:38:43,937] Trial 0 finished with value: 0.7246474839432138 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7246474839432138.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:44,450] Trial 1 finished with value: 0.7254912425040756 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7254912425040756.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:44,852] Trial 2 finished with value: 0.724716048705626 and parameters: {'l1_ratio': 0.22692876841884668}. 

Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:51,395] Trial 24 finished with value: 0.72551286145463 and parameters: {'l1_ratio': 0.21736087814603194}. Best is trial 12 with value: 0.72551286145463.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:51,629] Trial 25 finished with value: 0.7254912425040756 and parameters: {'l1_ratio': 0.29520653393380936}. Best is trial 12 with value: 0.72551286145463.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:51,828] Trial 26 finished with value: 0.6825802991707988 and parameters: {'l1_ratio': 0.06710037827301571}. Best is trial 12 with value: 0.72551286145463.
Fold 1 C-index: 0.625498

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7682403433476395
[I 2024-04-22 02:38:57,284] Trial 48 finished with value: 0.7237964201134265 and parameters: {'l1_ratio': 0.8619031648167028}. Best is trial 12 with value: 0.72551286145463.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.7021276595744681
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:57,637] Trial 49 finished with value: 0.7246401786742884 and parameters: {'l1_ratio': 0.3922995546354239}. Best is trial 12 with value: 0.72551286145463.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 02:38:57,915] Trial 50 finished with value: 0.6825802991707988 and parameters: {'l1_ratio': 0.04929515585590637}. B

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.22257178566632613
Fold 2 IBS: 0.1820993520215423
Fold 3 IBS: 0.19824420949810542
Fold 4 IBS: 0.2237600380590431
Fold 5 IBS: 0.16835053901639
[I 2024-04-22 02:39:16,602] Trial 0 finished with value: 0.19900518485228139 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.19900518485228139.
Fold 1 IBS: 0.2227354190136121
Fold 2 IBS: 0.18213858361607235
Fold 3 IBS: 0.19817288921098417
Fold 4 IBS: 0.223810434637058
Fold 5 IBS: 0.168114353842255
[I 2024-04-22 02:39:16,923] Trial 1 finished with value: 0.19899433606399633 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.19899433606399633.
Fold 1 IBS: 0.22281124531110838
Fold 2 IBS: 0.1821437908934777
Fold 3 IBS: 0.19816656925728893
Fold 4 IBS: 0.2238096257112818
Fold 5 IBS: 0.16807007886952047
[I 2024-04-22 02:39:17,282] Trial 2 finished with value: 0.19900026200853546 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value: 0.19899433606399633.
Fo

Fold 1 IBS: 0.22276760247931948
Fold 2 IBS: 0.18213554366252352
Fold 3 IBS: 0.19820478433293948
Fold 4 IBS: 0.22373156244662298
Fold 5 IBS: 0.16811705989773507
[I 2024-04-22 02:39:26,607] Trial 25 finished with value: 0.19899131056382807 and parameters: {'l1_ratio': 0.29603955452978525}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.22292920550164558
Fold 2 IBS: 0.1821559687414617
Fold 3 IBS: 0.2288717925709272
Fold 4 IBS: 0.2362661042284526
Fold 5 IBS: 0.16801617794830886
[I 2024-04-22 02:39:26,972] Trial 26 finished with value: 0.20764784979815923 and parameters: {'l1_ratio': 0.06084361560373101}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.22261093088211753
Fold 2 IBS: 0.18210937093302623
Fold 3 IBS: 0.19823058061874096
Fold 4 IBS: 0.2237467678392062
Fold 5 IBS: 0.16829739322937062
[I 2024-04-22 02:39:27,339] Trial 27 finished with value: 0.1989990087004923 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 17 with value: 0.1989526378

Fold 1 IBS: 0.22259390568462686
Fold 2 IBS: 0.18210751576992903
Fold 3 IBS: 0.1982566064103785
Fold 4 IBS: 0.22377442069288553
Fold 5 IBS: 0.1683168168243539
[I 2024-04-22 02:39:36,329] Trial 50 finished with value: 0.19900985307643476 and parameters: {'l1_ratio': 0.6064718647459368}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.22284765568252515
Fold 2 IBS: 0.18214842371985313
Fold 3 IBS: 0.198137752364713
Fold 4 IBS: 0.22378693605406708
Fold 5 IBS: 0.16802806434961035
[I 2024-04-22 02:39:36,760] Trial 51 finished with value: 0.19898976643415373 and parameters: {'l1_ratio': 0.1778917020771571}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.2228978928621057
Fold 2 IBS: 0.18215506607257112
Fold 3 IBS: 0.19809233332009074
Fold 4 IBS: 0.2237535021189615
Fold 5 IBS: 0.16797451868082638
[I 2024-04-22 02:39:37,170] Trial 52 finished with value: 0.19897466261091107 and parameters: {'l1_ratio': 0.10232328574307577}. Best is trial 17 with value: 0.1989526378

Fold 2 IBS: 0.18214532413755527
Fold 3 IBS: 0.19812857941594306
Fold 4 IBS: 0.22377329930093534
Fold 5 IBS: 0.16806019902960562
[I 2024-04-22 02:39:46,793] Trial 75 finished with value: 0.19897728993807168 and parameters: {'l1_ratio': 0.21877076880955348}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.22282224329850292
Fold 2 IBS: 0.18214780331796535
Fold 3 IBS: 0.19808138724048155
Fold 4 IBS: 0.2237331498149861
Fold 5 IBS: 0.16802815272447058
[I 2024-04-22 02:39:47,203] Trial 76 finished with value: 0.19896254727928128 and parameters: {'l1_ratio': 0.16913687383798529}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.22281896378671234
Fold 2 IBS: 0.18214760066133354
Fold 3 IBS: 0.1980879571489982
Fold 4 IBS: 0.22373942725583468
Fold 5 IBS: 0.16802969603156295
[I 2024-04-22 02:39:47,629] Trial 77 finished with value: 0.19896472897688836 and parameters: {'l1_ratio': 0.17011313497289468}. Best is trial 17 with value: 0.19895263780599762.
Fold 1 IBS: 0.247

In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.726
train_ibs:  0.199


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.21060026289682426)

test_cindex : 0.559


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.13474554059223504)

test_ibs:  0.29


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-22 02:39:57,315] A new study created in memory with name: no-name-4e9ef5c9-1cb2-4772-b20c-d2648f420393


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.6510638297872341
Fold 4 C-index: 0.752851711026616
Fold 5 C-index: 0.6909871244635193
[I 2024-04-22 02:40:01,577] Trial 0 finished with value: 0.6991621322401306 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6991621322401306.
Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7510729613733905
[I 2024-04-22 02:40:04,339] Trial 1 finished with value: 0.7355301592065651 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 4 C-index: 0.8022813688212928
Fold 5 C-index: 0.759656652360515
[I 2024-04-22 02:40:38,833] Trial 15 finished with value: 0.7430557612306835 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 114, 'oob_score': True, 'max_samples': 0.3692598016141903, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.17993556917544976, 'warm_start': True}. Best is trial 14 with value: 0.7726196368380054.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 02:40:38,993] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 6, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1937382585942945, 'warm_start': True}. Best is trial 14 with value: 0.7726196368380054.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.8914728682170543
Fold 3 C-index: 0.9276595744680851
Fold 4 C-index: 0.8935361216730038
Fold 5 C-index: 0.8969957081545065
[I 2024-04-22 02:40:49,243] Trial 31 finished with value: 0.8438452051001395 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 166, 'oob_score': True, 'max_samples': 0.8383718236388968, 'max_features': None, 'min_weight_fraction_leaf': 0.05349821135610229, 'warm_start': True}. Best is trial 30 with value: 0.8471808956568762.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.8875968992248062
Fold 3 C-index: 0.9319148936170213
Fold 4 C-index: 0.8935361216730038
Fold 5 C-index: 0.8969957081545065
[I 2024-04-22 02:40:50,452] Trial 32 finished with value: 0.8463115133784891 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 165, 'oob_score': True, 'max_samples': 0.844850651135347, 'max

...

Fold 5 IBS: 0.17585992091527558
[I 2024-04-22 02:43:23,359] Trial 31 finished with value: 0.19271571153723607 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 374, 'oob_score': False, 'max_samples': 0.7600957156160437, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.045073432551245296}. Best is trial 30 with value: 0.19201902148306207.
Fold 1 IBS: 0.22236771464371266
Fold 2 IBS: 0.17701546684229433
Fold 3 IBS: 0.19851905347240079
Fold 4 IBS: 0.18732302760178435
Fold 5 IBS: 0.18167933628393707
[I 2024-04-22 02:43:26,569] Trial 32 finished with value: 0.19338091976882582 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 394, 'oob_score': False, 'max_samples': 0.6752379846937863, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07718385932974292}. Best is trial 30 with value: 0.19201902148306207.
Fold 1 IBS: 0.2270170034083123
Fold 2 IBS: 0.17

Fold 1 IBS: 0.22670665484519142
Fold 2 IBS: 0.175184981867077
Fold 3 IBS: 0.20276219990812896
Fold 4 IBS: 0.19658107727261281
Fold 5 IBS: 0.17640448174832232
[I 2024-04-22 02:44:11,426] Trial 47 finished with value: 0.1955278791282665 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 14, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.7533756239183884, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.05433383705425634}. Best is trial 41 with value: 0.19046423845342159.
Fold 1 IBS: 0.21833683794653208
Fold 2 IBS: 0.16845072186321247
Fold 3 IBS: 0.20168040343516955
Fold 4 IBS: 0.1937674426294509
Fold 5 IBS: 0.19051837676249267
[I 2024-04-22 02:44:12,242] Trial 48 finished with value: 0.19455075652737155 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 82, 'oob_score': False, 'max_samples': 0.7075209195984526, 'max_features': 'auto', 'min_weight_fraction_leaf

Fold 5 IBS: 0.16839194066878888
[I 2024-04-22 02:44:47,192] Trial 62 finished with value: 0.1913839096782827 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 4, 'max_depth': 6, 'n_estimators': 298, 'oob_score': False, 'max_samples': 0.8075940187858324, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0028869935676452026}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22802304509837604
Fold 2 IBS: 0.17093887708958988
Fold 3 IBS: 0.19997810028764312
Fold 4 IBS: 0.19416891669551373
Fold 5 IBS: 0.16864914160971897
[I 2024-04-22 02:44:50,692] Trial 63 finished with value: 0.19235161615616833 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 305, 'oob_score': False, 'max_samples': 0.899801190989709, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0075083578814244606}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22706249643509485
Fold 2 IBS: 0.

Fold 1 IBS: 0.22661384896598818
Fold 2 IBS: 0.20671937958697834
Fold 3 IBS: 0.22631312111115084
Fold 4 IBS: 0.21497952798665046
Fold 5 IBS: 0.21282749449450544
[I 2024-04-22 02:45:35,672] Trial 78 finished with value: 0.21749067442905465 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 6, 'n_estimators': 289, 'oob_score': False, 'max_samples': 0.9147835512625551, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.43785214928176214}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.23052477127318172
Fold 2 IBS: 0.16982397129659127
Fold 3 IBS: 0.197745679074635
Fold 4 IBS: 0.19675496948041768
Fold 5 IBS: 0.1770213286573074
[I 2024-04-22 02:45:38,793] Trial 79 finished with value: 0.1943741439564266 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 7, 'n_estimators': 275, 'oob_score': False, 'max_samples': 0.9974228392350986, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.16901501941163646
[I 2024-04-22 02:46:11,875] Trial 93 finished with value: 0.19108673519318098 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 179, 'oob_score': False, 'max_samples': 0.8387093701884653, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.030817946786640082}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.22184856220737365
Fold 2 IBS: 0.17312992964385657
Fold 3 IBS: 0.1948930255496256
Fold 4 IBS: 0.1925411854591036
Fold 5 IBS: 0.1730631403095586
[I 2024-04-22 02:46:13,589] Trial 94 finished with value: 0.1910951686339036 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 182, 'oob_score': False, 'max_samples': 0.688472014648766, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03451598429298541}. Best is trial 49 with value: 0.19023957913930975.
Fold 1 IBS: 0.24047052597810953
Fold 2 IBS: 0.17250441

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.866
train_ibs:  0.19


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [58]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=10, max_leaf_nodes=14,
                     max_samples=0.8746458616251406, min_samples_leaf=2,
                     min_weight_fraction_leaf=0.030922977401692878,
                     n_estimators=113, random_state=123, warm_start=True)

test_cindex:  0.546


RandomSurvivalForest(max_depth=8, max_leaf_nodes=17,
                     max_samples=0.7890788621794695, min_samples_leaf=2,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.04458702793409107,
                     n_estimators=215, random_state=123)

test_ibs:  0.263


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [86]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [87]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-22 09:25:08,166] A new study created in memory with name: no-name-c0213e53-6473-4df9-a859-e07e85248eb4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.8553191489361702
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.7639484978540773
[I 2024-04-22 09:25:09,031] Trial 0 finished with value: 0.7623102721396274 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7623102721396274.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 09:25:10,678] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.6759656652360515
[I 2024-04-22 09:25:28,399] Trial 16 finished with value: 0.7032137338705444 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7867972827001483.
Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.7718631178707225
Fold 5 C-index: 0.6824034334763949
[I 2024-04-22 09:25:28,929] Trial 17 finished with value: 0.731554986493769 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.649402390438247
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.8068669527896996
[I 2024-04-22 09:25:40,509] Trial 31 finished with value: 0.7651208885111453 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.6357896182918609, 'min_weight_fraction_leaf': 0.043457520259011145}. Best is trial 23 with value: 0.7937218312264874.
Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7286821705426356
Fold 3 C-index: 0.8340425531914893
Fold 4 C-index: 0.7870722433460076
Fold 5 C-index: 0.7553648068669528
[I 2024-04-22 09:25:41,179] Trial 32 finished with value: 0.7493192073790584 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features': 1,

...

Fold 1 IBS: 0.2233579319393271
Fold 2 IBS: 0.20096408163499427
Fold 3 IBS: 0.19049881866793233
Fold 4 IBS: 0.20774755146340546
Fold 5 IBS: 0.19123419554925844
[I 2024-04-22 09:27:56,893] Trial 30 finished with value: 0.20276051585098354 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.19459187877865486.
Fold 1 IBS: 0.2179837541713329
Fold 2 IBS: 0.19996868148399857
Fold 3 IBS: 0.18950194352113112
Fold 4 IBS: 0.20402491486791338
Fold 5 IBS: 0.18472430994108335
[I 2024-04-22 09:27:59,279] Trial 31 finished with value: 0.19924072079709187 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.22018361682359297
Fold 2 IBS: 0.19767576801393197
Fold 3 IBS: 0.19421270363305465
Fold 4 IBS: 0.20406059613721392
Fold 5 IBS: 0.1854331539519078
[I 2024-04-22 09:28:43,976] Trial 45 finished with value: 0.20031316771194024 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 14, 'n_estimators': 477, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8420791678401572, 'min_weight_fraction_leaf': 0.1114478053387663}. Best is trial 34 with value: 0.1915510180929927.
Fold 1 IBS: 0.22677495195617972
Fold 2 IBS: 0.20938982706068335
Fold 3 IBS: 0.20301008325030304
Fold 4 IBS: 0.21538736174154596
Fold 5 IBS: 0.2016448459853074
[I 2024-04-22 09:28:47,512] Trial 46 finished with value: 0.21124141399880392 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 432, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.945

Fold 1 IBS: 0.21476513837056027
Fold 2 IBS: 0.18143050987640613
Fold 3 IBS: 0.1908356407571883
Fold 4 IBS: 0.19427180424062881
Fold 5 IBS: 0.16683442835564766
[I 2024-04-22 09:29:36,677] Trial 60 finished with value: 0.18962750432008624 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 439, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8952499671372233, 'min_weight_fraction_leaf': 0.036398817545426995}. Best is trial 60 with value: 0.18962750432008624.
Fold 1 IBS: 0.21627361369698
Fold 2 IBS: 0.1817225137067132
Fold 3 IBS: 0.19073393689024337
Fold 4 IBS: 0.19494098781318964
Fold 5 IBS: 0.16750297353174665
[I 2024-04-22 09:29:40,190] Trial 61 finished with value: 0.19023480512777455 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 438, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.87

Fold 1 IBS: 0.21657675292627207
Fold 2 IBS: 0.18996174454319362
Fold 3 IBS: 0.18393478555155848
Fold 4 IBS: 0.19722264590644567
Fold 5 IBS: 0.17340727825186003
[I 2024-04-22 09:30:22,125] Trial 75 finished with value: 0.19222064143586598 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 391, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9086795540810024, 'min_weight_fraction_leaf': 0.03284410208925504}. Best is trial 63 with value: 0.1883299130608305.
Fold 1 IBS: 0.21374362171322053
Fold 2 IBS: 0.1809837734102344
Fold 3 IBS: 0.1928804193379134
Fold 4 IBS: 0.19389862294434004
Fold 5 IBS: 0.16641872761123616
[I 2024-04-22 09:30:25,853] Trial 76 finished with value: 0.1895850330033889 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 417, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8

Fold 1 IBS: 0.21649654230068696
Fold 2 IBS: 0.1994536066038553
Fold 3 IBS: 0.1910443715333566
Fold 4 IBS: 0.20311940988854024
Fold 5 IBS: 0.18412454653280888
[I 2024-04-22 09:31:00,193] Trial 90 finished with value: 0.1988476953718496 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 305, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9531106840561406, 'min_weight_fraction_leaf': 0.09026278115621361}. Best is trial 63 with value: 0.1883299130608305.
Fold 1 IBS: 0.21417020956953955
Fold 2 IBS: 0.18221070479947724
Fold 3 IBS: 0.1876123093586995
Fold 4 IBS: 0.1945453292939215
Fold 5 IBS: 0.1658337819099612
[I 2024-04-22 09:31:03,452] Trial 91 finished with value: 0.1888744669863198 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 13, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 390, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.915582

In [88]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [89]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.188


#### Test

In [90]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [91]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_leaf_nodes=18,
                   max_samples=0.8161032428574773, min_samples_leaf=2,
                   min_samples_split=11,
                   min_weight_fraction_leaf=0.001106761767719347,
                   n_estimators=105, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.607


ExtraSurvivalTrees(max_depth=10, max_features=None, max_leaf_nodes=12,
                   max_samples=0.8664901703822724, min_samples_split=13,
                   min_weight_fraction_leaf=0.0002786019075411844,
                   n_estimators=373, oob_score=True, random_state=123)

IBS: 0.252


In [92]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-22 02:56:57,588] A new study created in memory with name: no-name-9f783f1b-e69e-4451-a4e0-312e861a38aa


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 02:57:31,250] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 02:57:49,205] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 03:06:06,582] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7053891693443892.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 03:07:07,590] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 03:13:51,808] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7053891693443892.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-22 03:14:10,603] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-22 04:02:45,700] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:03:21,802] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.35760

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:08:21,903] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:08:37,060] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-22 04:13:24,633] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 22 with value: 0.23482204102612672.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:13:41,094] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:18:40,155] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23331347435775504.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-22 04:19:22,401] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-22 04:25:50,010] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23331347435775504.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23331347435775504], datetime_start=datetime.datetime(2024, 4, 22, 4, 16, 19, 377434), datetime_complete=datetime.datetime(2024, 4, 22, 4, 16, 59, 948312), params={'subsample': 0.9481727373988897, 'learning_rate': 0.020229752590967

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.759
train_ibs:  0.233


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.008821356078039954,
                                 criterion='squared_error',
                                 dropout_rate=0.5604101432191428,
                                 learning_rate=0.05159117661647313, max_depth=4,
                                 max_features=0.1, max_leaf_nodes=18,
                                 min_impurity_decrease=9.803482393630746e-05,
                                 min_samples_leaf=7, min_samples_split=8,
                                 min_weight_fraction_leaf=0.26023745967507866,
                                 n_estimators=350, random_state=123,
                                 subsample=0.9197287558651637,
                                 validation_fraction=0.8247616419720114)

C-index score: 0.586


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.228


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-22 04:26:01,694] A new study created in memory with name: no-name-62a2374e-892f-4ac4-bb03-f70c39255f9c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-22 04:26:02,446] Trial 0 finished with value: 0.6249682148055231 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6566523605150214
[I 2024-04-22 04:26:08,267] Trial 1 finished with value: 0.6241098457068107 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6781115879828327
[I 2024-04-22 04:26:57,923] Trial 19 finished with value: 0.6399834362571664 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.6546369717001154.
Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6738197424892703
[I 2024-04-22 04:27:02,783] Trial 20 finished with value: 0.6375499035301421 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.6546369717001154.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.5659574468085107
Fold 4 C-index: 0.6806083650190115

Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6738197424892703
[I 2024-04-22 04:27:48,984] Trial 38 finished with value: 0.6382987793600792 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.7668386839679121, 'n_estimators': 46, 'learning_rate': 0.06351044805702724}. Best is trial 11 with value: 0.6546369717001154.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6695278969957081
[I 2024-04-22 04:27:49,975] Trial 39 finished with value: 0.6297198966721359 and parameters: {'subsample': 0.3132356181950913, 'dropout_rate': 0.5267513165392328, 'n_estimators': 159, 'learning_rate': 0.05598015324329953}. Best is trial 11 with value: 0.6546369717001154.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fold 5 C-index: 0.6609442060085837
[I 2024-04-22 04:27:50,376] Trial 40

...

Fold 4 IBS: 0.2426706185564074
Fold 5 IBS: 0.22283081379821795
[I 2024-04-22 04:31:00,772] Trial 19 finished with value: 0.24310221296462103 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.24550608887605468
Fold 2 IBS: 0.2133505584021398
Fold 3 IBS: 0.23294747738469943
Fold 4 IBS: 0.23024731593258208
Fold 5 IBS: 0.2123372739360029
[I 2024-04-22 04:31:01,157] Trial 20 finished with value: 0.22687774290629575 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2448259892509062
Fold 2 IBS: 0.21786672573833757
Fold 3 IBS: 0.23050391968540718
Fold 4 IBS: 0.23249100529840805
Fold 5 IBS: 0.215533050087791
[I 2024-04-22 04:31:01,529] Trial 21 finished with value: 0.2282441380121

Fold 4 IBS: 0.23453614765450895
Fold 5 IBS: 0.2122914778844625
[I 2024-04-22 04:31:17,584] Trial 38 finished with value: 0.2333975796328732 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2943463281154152
Fold 2 IBS: 0.21435275898375683
Fold 3 IBS: 0.292776623523439
Fold 4 IBS: 0.2606234117164125
Fold 5 IBS: 0.24191548716779984
[I 2024-04-22 04:31:17,957] Trial 39 finished with value: 0.26080292190136467 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2956705905294083
Fold 2 IBS: 0.21430445848659016
Fold 3 IBS: 0.2928588854990794
Fold 4 IBS: 0.2596622123064148
Fold 5 IBS: 0.24678618384472736
[I 2024-04-22 04:31:18,559] Trial 40 finished with value: 0.261856466133244 a

Fold 5 IBS: 0.27461860801929316
[I 2024-04-22 04:31:28,033] Trial 57 finished with value: 0.29238736666151666 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2524958013314946
Fold 2 IBS: 0.20467957580837548
Fold 3 IBS: 0.24466604990129345
Fold 4 IBS: 0.22951388023914565
Fold 5 IBS: 0.21232254648299242
[I 2024-04-22 04:31:28,356] Trial 58 finished with value: 0.22873557075266032 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2450766728615442
Fold 2 IBS: 0.21570571438138603
Fold 3 IBS: 0.23170031253784298
Fold 4 IBS: 0.23100841297079955
Fold 5 IBS: 0.21387190006582926
[I 2024-04-22 04:31:28,572] Trial 59 finished with value: 0.2274726025634804 and parameters: {'subsample

Fold 1 IBS: 0.2506811440240583
Fold 2 IBS: 0.2053539697592853
Fold 3 IBS: 0.2432701864482772
Fold 4 IBS: 0.2284683602008423
Fold 5 IBS: 0.21006566848816424
[I 2024-04-22 04:31:46,127] Trial 77 finished with value: 0.22756786578412544 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.24728609182637104
Fold 2 IBS: 0.20948456237788454
Fold 3 IBS: 0.2367879689804507
Fold 4 IBS: 0.22900068533758589
Fold 5 IBS: 0.21042389725248523
[I 2024-04-22 04:31:46,541] Trial 78 finished with value: 0.2265966411549555 and parameters: {'subsample': 0.5939500427136554, 'dropout_rate': 0.7406477151463077, 'n_estimators': 73, 'learning_rate': 0.015420072941943445}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2491522359439618
Fold 2 IBS: 0.20720785646863638
Fold 3 IBS: 0.23970965460068624
Fold 4 IBS: 0.22880108027481874
Fold 5 IB

Fold 1 IBS: 0.3528807852002724
Fold 2 IBS: 0.26163876419301324
Fold 3 IBS: 0.35927296600690356
Fold 4 IBS: 0.2978922662575483
Fold 5 IBS: 0.3041795583962892
[I 2024-04-22 04:31:55,721] Trial 96 finished with value: 0.31517286801080535 and parameters: {'subsample': 0.41313939490511475, 'dropout_rate': 0.5213109482382285, 'n_estimators': 158, 'learning_rate': 0.054487916689644846}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.24708172105946397
Fold 2 IBS: 0.21005062605084635
Fold 3 IBS: 0.23658239945020892
Fold 4 IBS: 0.2291089640988238
Fold 5 IBS: 0.2105403389046395
[I 2024-04-22 04:31:56,148] Trial 97 finished with value: 0.2266728099127965 and parameters: {'subsample': 0.5389995281019297, 'dropout_rate': 0.6077476353900257, 'n_estimators': 73, 'learning_rate': 0.014931416501109941}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.26183286577938214
Fold 2 IBS: 0.20183342233582774
Fold 3 IBS: 0.2575253907596287
Fold 4 IBS: 0.23417362585495322
Fold 5 IBS:

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.655
train_ibs:  0.227


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10580803679784617,
                                              learning_rate=0.08933703080421439,
                                              n_estimators=324,
                                              random_state=123,
                                              subsample=0.11557957726835853)

C-index score: 0.536


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6341528845468055,
                                              learning_rate=0.014291926646619518,
                                              n_estimators=76, random_state=123,
                                              subsample=0.6070545538128025)

IBS: 0.234


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [93]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.866,1.0
ExtraSurvivalTrees,0.826,2.0
GradientBoosting,0.759,3.0
CoxElastic,0.726,4.0
CoxLasso,0.725,5.0
CoxPH,0.724,6.0
ComponentwiseGradientBoosting,0.655,7.0
CoxRidge,0.649,8.0


In [94]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.188,1.0
Randomsurvivalforest,0.190,2.0
CoxLasso,0.199,3.5
CoxElastic,0.199,3.5
CoxPH,0.200,5.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [95]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.607,1.0
GradientBoosting,0.586,2.0
CoxPH,0.562,3.0
CoxLasso,0.561,4.0
CoxElastic,0.559,5.0
Randomsurvivalforest,0.546,6.0
CoxRidge,0.537,7.0
ComponentwiseGradientBoosting,0.536,8.0


In [96]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
GradientBoosting,0.228,1.0
CoxRidge,0.229,2.0
ComponentwiseGradientBoosting,0.234,3.0
ExtraSurvivalTrees,0.252,4.0
Randomsurvivalforest,0.263,5.0
CoxLasso,0.289,6.0
CoxPH,0.290,7.5
CoxElastic,0.290,7.5


In [97]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/standard/rent/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [98]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-22
